# UCS761 - Deep Learning Lab 2
### From Hard Decisions to Calibrated Decisions

Lab 1 showed that a perceptron outputs either 0 or 1, nothing in between.
The problem is that some samples sit very close to the decision boundary. A tiny bit of noise can flip the decision completely.

This lab is about swapping the step function out for sigmoid, so the model gives a probability instead of a hard answer. That probability tells you how confident the model actually is.

In [1]:
import numpy as np
import math


## The Problem with Hard Decisions

Take sodium content in glass as an example. If the threshold is 13.5 and a sample measures 13.5 ± 0.3 because of measurement noise, the decision can flip back and forth randomly.

A hard YES or NO answer gives you no idea how sure the model is.

In [2]:
sodium = [12.8, 13.0, 13.2, 13.4, 13.5, 13.6, 13.8, 14.0, 14.2, 14.5]
labels = [  0,    0,    0,    0,    1,    0,    1,    1,    1,    1  ]

w_hard = 1.0
b_hard = -13.5

print("Step function decisions:")
for na, y in zip(sodium, labels):
    z   = w_hard * na + b_hard
    out = 1 if z >= 0 else 0
    print(f"  Na={na:.1f}  ->  predicted={out}  actual={y}")


Step function decisions:
  Na=12.8  ->  predicted=0  actual=0
  Na=13.0  ->  predicted=0  actual=0
  Na=13.2  ->  predicted=0  actual=0
  Na=13.4  ->  predicted=0  actual=0
  Na=13.5  ->  predicted=1  actual=1
  Na=13.6  ->  predicted=1  actual=0
  Na=13.8  ->  predicted=1  actual=1
  Na=14.0  ->  predicted=1  actual=1
  Na=14.2  ->  predicted=1  actual=1
  Na=14.5  ->  predicted=1  actual=1


## Noise Problem Demonstrated

When a borderline sample has slight noise added, the step function flips completely.

In [5]:
borderline = 13.5
noise_range = [-0.3, -0.1, 0.0, 0.1, 0.3]

print("What happens at Na=13.5 with measurement noise:")
for noise in noise_range:
    na_val = borderline + noise
    z      = w_hard * na_val + b_hard
    out    = 1 if z >= 0 else 0
    print(f"  Na={na_val:.2f}  ->  output={out}")

print("Same sample, different noise — completely different decisions. Not ideal.")


What happens at Na=13.5 with measurement noise:
  Na=13.20  ->  output=0
  Na=13.40  ->  output=0
  Na=13.50  ->  output=1
  Na=13.60  ->  output=1
  Na=13.80  ->  output=1
Same sample, different noise — completely different decisions. Not ideal.


## Sigmoid Function

Sigmoid smooths out the decision. Instead of jumping from 0 to 1 at z=0, it transitions gradually.

Formula: sigma(z) = 1 / (1 + e^(-z))

Key points:
- z = 0 gives 0.5 (maximum uncertainty)
- large positive z gives output close to 1
- large negative z gives output close to 0

In [6]:
def sigmoid(z):
    return 1 / (1 + math.exp(-z))

print("Sigmoid values at key points:")
for z in [-5, -2, -1, 0, 1, 2, 5]:
    print(f"  sigma({z:3d}) = {sigmoid(z):.4f}")


Sigmoid values at key points:
  sigma( -5) = 0.0067
  sigma( -2) = 0.1192
  sigma( -1) = 0.2689
  sigma(  0) = 0.5000
  sigma(  1) = 0.7311
  sigma(  2) = 0.8808
  sigma(  5) = 0.9933


In [7]:
print("Sigmoid predictions on sodium data:")
for na, y in zip(sodium, labels):
    z    = w_hard * na + b_hard
    prob = sigmoid(z)
    dec  = 1 if prob >= 0.5 else 0
    print(f"  Na={na:.1f}  prob={prob:.4f}  decision={dec}  actual={y}")


Sigmoid predictions on sodium data:
  Na=12.8  prob=0.3318  decision=0  actual=0
  Na=13.0  prob=0.3775  decision=0  actual=0
  Na=13.2  prob=0.4256  decision=0  actual=0
  Na=13.4  prob=0.4750  decision=0  actual=0
  Na=13.5  prob=0.5000  decision=1  actual=1
  Na=13.6  prob=0.5250  decision=1  actual=0
  Na=13.8  prob=0.5744  decision=1  actual=1
  Na=14.0  prob=0.6225  decision=1  actual=1
  Na=14.2  prob=0.6682  decision=1  actual=1
  Na=14.5  prob=0.7311  decision=1  actual=1


## Sigmoid vs Step on Noisy Input

Same borderline sample with noise: sigmoid changes gradually, step function flips abruptly.

In [9]:
print("Noisy Na=13.5 -- Sigmoid vs Step:")
for noise in noise_range:
    na_val   = borderline + noise
    z        = w_hard * na_val + b_hard
    sig_out  = sigmoid(z)
    step_out = 1 if z >= 0 else 0
    print(f"  Na={na_val:.2f}  sigmoid={sig_out:.4f}  step={step_out}")

print("Sigmoid changes gradually. Step function flips abruptly.")


Noisy Na=13.5 -- Sigmoid vs Step:
  Na=13.20  sigmoid=0.4256  step=0
  Na=13.40  sigmoid=0.4750  step=0
  Na=13.50  sigmoid=0.5000  step=1
  Na=13.60  sigmoid=0.5250  step=1
  Na=13.80  sigmoid=0.5744  step=1
Sigmoid changes gradually. Step function flips abruptly.


## Threshold as a Policy

The model gives a probability. The threshold is your decision policy. It is completely separate from the model itself.
You can tune it based on what kind of errors are more costly in your application.

In [10]:
def decide(na_list, w, b, threshold):
    results = []
    for na in na_list:
        z = w * na + b
        p = sigmoid(z)
        results.append(1 if p >= threshold else 0)
    return results

pred_0_5 = decide(sodium, w_hard, b_hard, 0.5)
pred_0_7 = decide(sodium, w_hard, b_hard, 0.7)

print("Threshold = 0.5:", pred_0_5)
print("Threshold = 0.7:", pred_0_7)
print("Actual:         ", labels)


Threshold = 0.5: [0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
Threshold = 0.7: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Actual:          [0, 0, 0, 0, 1, 0, 1, 1, 1, 1]


## Why 0.7 is Safer for Glass Quality Control

In glass manufacturing, accepting a defective piece (false positive) costs way more than rejecting a good one.

With threshold = 0.7, the model only marks something as Type 1 when it is 70% or more confident. Fewer false acceptances, which is the right trade-off when the stakes are high.

**Summary:**
Step function is fine when data is clean and well-separated.
Near the boundary, noise causes instability.
Sigmoid gives a probability, not just a binary answer.
The threshold is a policy decision, not part of the model.
Higher threshold means more conservative, fewer false positives.